In [3]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [4]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


In [9]:
print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER GLOBAL - FASE 3 (SALING SILANG & AUTO-SKIP) 🚀 ")
print("================================================================================")

# ================================================================================
# TAHAP 1: MEMUAT SE LURUH BERKAS PICKLE DAN MERGE JADI SATU PINTU
# ================================================================================
all_fase_3_data = {}

# 1. Load File Cimut
try:
    with open('fase_3_cimut.pkl', 'rb') as f:
        all_fase_3_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Cimut.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_3_afrida.pkl', 'rb') as f:
        all_fase_3_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Afrida.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif (Jika ada file terpisah, opsional)
try:
    if os.path.exists('fase_3_hanif.pkl'):
        with open('fase_3_hanif.pkl', 'rb') as f:
            all_fase_3_data.update(pickle.load(f))
        print("✓ Berhasil memuat data hasil konversi Hanif.")
except:
    print(f"⚠️ Gagal memuat file pkl Hanif: {f}")
    pass

 🚀 SETUP INSERT HANDLER GLOBAL - FASE 3 (SALING SILANG & AUTO-SKIP) 🚀 
✓ Berhasil memuat data hasil konversi Cimut.
✓ Berhasil memuat data hasil konversi Afrida.
⚠️ Gagal memuat file pkl Hanif: <_io.BufferedReader name='fase_3_hanif.pkl'>


In [10]:
# ================================================================================
# TAHAP 2: ATUR URUTAN STRATEGIS SALING SILANG (CIMUT, AFRIDA, & HANIF)
# ================================================================================
# Susunan di bawah ini diatur ketat lintas personel agar Foreign Key aman masuk ke MySQL!
tables_to_insert_ordered = [
    # --- BLOK A: PENDAFTARAN & SDM (Karya Hanif) ---
    'pelamar',                  # Induk data pelamar kerja/kursus
    'pelamar_kerja',            # Detail pelamar posisi kerja
    'pelamar_sekolah',          # Riwayat sekolah pelamar
    'pelamar_kursus',           # Riwayat kursus pelamar
    'progres_pelamar',          # Log catatan tahapan seleksi
    'rekrutmen_pelamar',        # Keputusan akhir rekrutmen pelamar
    'pengajuan_karyawan',       # Form pengajuan penambahan staff baru
    'histori_pengajuan',        # Log alur persetujuan pengajuan staff

    # --- BLOK B: SURAT-MENYURAT & SOP (Karya Afrida) ---
    'sop',                      # Standar operasional prosedur instansi
    'surat_keluar',             # Log keluar dokumen/surat resmi
    'verifikasi_surat_keluar',  # Log persetujuan surat keluar oleh atasan
    'surat_tugas',              # Surat perintah penugasan formal
    'surat_tugas_anggota',      # Anggota staff yang terikat di dalam surat tugas

    # --- BLOK C: MARKETING & ADMISI CALON SISWA (Karya Cimut) ---
    'kontak_prospek',           # Database mentah leads / prospek marketing
    'calon_siswa',              # Formulir profil utama calon siswa baru
    'calon_siswa_ortu',         # Data wali / orang tua calon siswa
    'calon_siswa_akademik',     # Riwayat background akademik calon siswa
    'calon_siswa_bayar',        # Log transaksi pembayaran formulir/DP awal
    'calon_siswa_jadwal',       # Plotting jadwal tes/interview calon siswa
    'calon_siswa_kursus',       # Pilihan program kursus yang diminati calon siswa
    'calon_siswa_proses',       # Jalur perkembangan dokumen admisinya
    'calon_siswa_status_logs',  # Log perubahan status akhir (Diterima/Ditolak/Pending)

    # --- BLOK D: LOGISTIK & OPERASIONAL INTERNAL (Karya Cimut) ---
    'pengadaan',                # Form pengajuan belanja/pengadaan aset barang
    'peminjaman',               # Log pinjam pakai sarana prasarana oleh staff
    'problem'                   # Log laporan kerusakan/kendala teknis fasilitas
]

In [11]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT DENGAN RINGKASAN DI ATAS & DIAGNOSTIK ERROR DI BAWAH
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DI BELAKANG LAYAR
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {
                'status': 'not_found', 
                'rows': 0, 
                'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl'
            }
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {
                'status': 'empty', 
                'rows': 0, 
                'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)'
            }
            continue
            
        try:
            # Bersihkan kolom kosong murni agar tidak merusak placeholder query
            df_to_push = df_target.dropna(axis=1, how='all')
            
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            
            # Gunakan INSERT IGNORE untuk auto-skip duplikat primary key
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            # Konversi DataFrame ke Native List Python (Hancurkan tipe data NumPy)
            raw_numpy_list = df_to_push.to_numpy().tolist()
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN"] else x for x in row) 
                for row in raw_numpy_list
            ]
            
            # Eksekusi massal
            cursor.executemany(insert_query, clean_data_tuples)
            db_connection.commit()
            
            total_rows = len(clean_data_tuples)
            results[table_name] = {
                'status': 'success', 
                'rows': total_rows, 
                'msg': f'✓ {table_name}: Sukses diproses! Sebanyak {total_rows} baris sukses dimasukkan / di-skip aman.'
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'rows': 0, 
                'msg': f'✗ {table_name}: Gagal total saat insert - Alasan: {e}'
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    # 1. Cetak yang sukses dulu biar rapi
    print("🟢 TABEL YANG SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] == 'success':
            print(f"  {results[table_name]['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses)")

    print("\n🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):")
    failed_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] in ['failed', 'not_found', 'empty']:
            print(f"  {results[table_name]['msg']}")
            failed_exist = True
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada yang gagal.")
            
    print("================================================================================\n")


    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            # Taktik A: Jika Sukses, tampilkan preview standard 5 baris teratas
            if results[table_name]['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name]) 
                print("-" * 80)
                
            # Taktik B: Jika GAGAL, tembak dan kuliti struktur datanya secara transparan!
            elif results[table_name]['status'] == 'failed':
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Alasan MySQL Menolak: {results[table_name]['msg']}")
                print("-" * 50)
                print("Berikut 5 baris sampel data yang gagal dikirim, cek tipe datanya (apakah ada .0 atau string aneh):")
                display(tables_data[table_name])
                print(f"\nTipe data kolom internal DataFrame untuk tabel '{table_name}':")
                # Menampilkan tipe data internal pandas agar ketahuan mana float ghaib / objek aneh
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

In [12]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL (MENGGUNAKAN VERSI RINGKASAN ATAS)
# ================================================================================
results_fase_3 = insert_data_with_preview_and_skip_v2(
    db_connection=db_new, 
    cursor=cursor_new, 
    tables_data=all_fase_3_data, 
    ordered_list=tables_to_insert_ordered
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)

 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG SUKSES MASUK:
  ✓ sop: Sukses diproses! Sebanyak 4 baris sukses dimasukkan / di-skip aman.
  ✓ surat_keluar: Sukses diproses! Sebanyak 213 baris sukses dimasukkan / di-skip aman.
  ✓ verifikasi_surat_keluar: Sukses diproses! Sebanyak 477 baris sukses dimasukkan / di-skip aman.
  ✓ surat_tugas: Sukses diproses! Sebanyak 135 baris sukses dimasukkan / di-skip aman.
  ✓ surat_tugas_anggota: Sukses diproses! Sebanyak 304 baris sukses dimasukkan / di-skip aman.
  ✓ kontak_prospek: Sukses diproses! Sebanyak 214 baris sukses dimasukkan / di-skip aman.
  ✓ calon_siswa: Sukses diproses! Sebanyak 214 baris sukses dimasukkan / di-skip aman.
  ✓ calon_siswa_ortu: Sukses diproses! Sebanyak 214 baris sukses dimasukkan / di-skip aman.
  ✓ calon_siswa_akademik: Sukses diproses! Sebanyak 214 baris sukses dimasukkan / di-skip aman.
  ✓ calon_siswa_bayar: Sukse

,judul_sop,link_dokumen_sop,created_at,id_sop_kategori
0,Akhir Penggunaan Kelas English (19.15 WIB),https://drive.google.com/file/d/1V0MpB7ctzPHe6...,2023-07-07 08:52:48,1
1,Penerimaan Surat Masuk,https://drive.google.com/file/d/1G8lO33yU7MufC...,2023-11-22 15:27:06,2
2,Pengajuan Surat Keluar,https://drive.google.com/file/d/14I7wYSk62WJAv...,2023-11-22 15:27:55,2
3,Pengajuan Surat Tugas,https://drive.google.com/file/d/1R-wKFqQeVSz62...,2023-11-22 15:28:27,2


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: SURAT_KELUAR]
--------------------------------------------------


,id_sk,id_user,keterangan_sk,link_dokumen_sk,status_sk,nomor_sk,catatan_sk,created_at
0,24,U00011,Surat Permohonan Uji Kompetensi dan Penggunaan...,https://docs.google.com/document/d/1xyhc4T-FlR...,Sudah Revisi,090A/LEAP/IX/2023- 090B/LEAP/IX/2023,Lengkapi Tanggal Pelaksanaan dengan tanggal ya...,2023-09-11 17:57:49
1,26,U00011,Beasiswa Siswa GE,https://docs.google.com/document/d/1jMTK_3b57e...,Disetujui,101/LEAP/BD/X/2023,,2023-10-25 16:57:38
2,27,U00011,PERJANJIAN KERJASAMA / MEMORANDUM OF UNDERSTAN...,https://docs.google.com/document/d/1Edeb7hWaBq...,Disetujui,102/LEAP/BD/X/2023,,2023-10-26 17:37:59
3,28,U00026,Sertifikat / Sertifikat Kelas APK Private / 1 ...,https://drive.google.com/drive/folders/1JMrPJF...,Disetujui,Sertif 002a/APEX/XI/2324/02 (Page 1) dan 002b/...,,2023-11-10 16:10:04
4,29,U00011,Penawaran Pelatihan Business English ke PT. La...,https://docs.google.com/document/d/1wBK62noEJw...,Disetujui,105/LEAP/BD/XI/2023,,2023-11-13 15:41:58
...,...,...,...,...,...,...,...,...
208,248,U00016,Jurnal Siswa & Laporan Akhir Community Service...,https://docs.google.com/document/d/1QJOQCHDN3j...,Disetujui,,,2026-04-07 14:44:54
209,250,U00033,Pelaporan Hasil Ujian Susulan Semester Genap K...,https://drive.google.com/file/d/14MFeTQ6SsupYk...,Disetujui,042/PDDK/SKET/LEAP/IV/2026,,2026-04-14 13:51:55
210,251,U00023,MoM Rupin (Mid Semester Evaluation & Next Batc...,https://docs.google.com/document/d/1s3G_jzBIlH...,Disetujui,043/PDDK/MoM/LEAP/IV/2026,,2026-04-16 08:51:31
211,252,U00034,Surat Izin Uji Coba Proyek SMPN 13,https://docs.google.com/document/d/14r8_bXEJjl...,Disetujui,046/PDDK/PM/LEAP/IV/2026,,2026-04-17 15:38:24


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: VERIFIKASI_SURAT_KELUAR]
--------------------------------------------------


...

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PEMINJAMAN]
--------------------------------------------------


,id_pinjam,tanggal_pinjam,keperluan,id_user,status_pinjam,catatan_sarpras,created_at
0,2,2023-06-11,<p>pinjam kamera - fun class tk mitra - 1 - 11...,U00026,Diajukan,None,2023-06-12 13:20:39
1,3,2023-06-11,<p>1. kamera - fun class TK mitra - 1 - 11 Jun...,U00026,Diajukan,None,2023-06-12 13:21:35
2,5,2023-08-21,<p>1. Tablet Leap - 1 - Jaga-jaga untuk Mid Te...,U00026,Diajukan,None,2023-08-16 15:01:55
3,6,2023-08-23,<p>Pinjam kamera untuk rekaman video checklist...,U00033,Diajukan,None,2023-08-23 10:19:12
4,7,2023-10-11,<p>1. Tablet Leap - 1 - Jaga-jaga untuk Final ...,U00026,Diajukan,None,2023-10-10 15:30:53
...,...,...,...,...,...,...,...
189,192,2026-04-10,"<p><span style=""color: #212529; font-family: R...",U00060,Diajukan,<p>Sudah dikembalikan</p>,2026-04-09 11:40:52
190,193,2026-04-10,"<p><span style=""color: #212529; font-family: R...",U00041,Diajukan,None,2026-04-10 16:26:19
191,194,2026-04-17,<p>List Peminjaman barang kegiatan student app...,U00060,Diajukan,None,2026-04-15 16:06:08
192,195,2026-04-17,"<p><span style=""color: #212529; font-family: R...",U00060,Diajukan,None,2026-04-16 15:22:28


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PROBLEM]
--------------------------------------------------


,id_problem,detail_masalah,id_user,status_perbaikan,tanggal_lapor,tanggal_selesai,catatan_teknisi,gambar_problem
0,43,AC Kelas Miss Erika kurang dingin ( belakang,U00012,Diajukan,2023-06-28 16:07:52,NaT,"sudah info ke Pak Irawan,\r\nTukang AC masih l...",None
1,58,Boya/mic untuk kelas hybrid tidak berfungsi (,U00026,Diajukan,2023-07-06 10:45:03,2023-07-17,None,None
2,60,tegangan listrik di ruang kelas belakang dapur...,U00033,Diajukan,2023-07-13 16:59:57,2023-08-10,pemberian stabilizer,None
3,61,ac brisik,U00033,Diajukan,2023-07-14 09:24:29,2023-07-25,sudah tidak berisik,None
4,62,Kabel power monitor PC room 2 longgar. Saat me...,U00036,Diajukan,2023-07-17 15:04:01,2023-07-17,None,None
...,...,...,...,...,...,...,...,...
155,251,"AC room 6 tidak dingin, dan ada air menetes da...",U00050,Diajukan,2026-04-08 16:55:36,2026-04-13,None,None
156,252,AC Ruang 6 (miss Peni) kondisi saat ini di OFF...,U00020,Diajukan,2026-04-09 16:31:44,2026-04-13,AC sudah diperbaiki,/storage/sarpas_images/sarpas_69d772008efe9.jpeg
157,253,Sesi 3. Hybrid. Guru menggunakan mic unt hybri...,U00026,Diajukan,2026-04-13 19:02:53,2026-04-15,None,/storage/sarpas_images/sarpas_69dcdb6dd7663.jpg
158,254,"Bracket TV kurang kenceng, suka geser kedepan ...",U00019,Diajukan,2026-04-16 16:58:21,2026-04-17,None,None


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [7]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 3 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_3 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )